# Structured JSON Extraction Fine-Tuning: LoRA + DPO

This notebook runs the full project on Google Colab Pro with an L4 GPU.

Pipeline:
- Install dependencies
- Generate the synthetic medical-note extraction dataset
- Run QLoRA SFT on Qwen2.5-3B
- Evaluate baseline vs SFT
- Generate DPO preference pairs with GPT-4o-mini
- Run DPO on top of the SFT adapter
- Re-evaluate and save final artifacts


## Runtime Notes

Before running:
- In Colab, choose `Runtime -> Change runtime type -> GPU`
- Recommended GPU: `L4`
- If you want DPO labeling in-notebook, add `OPENAI_API_KEY` as a Colab secret or set it manually below


In [ ]:
# Option 1: clone your repo
# !git clone https://github.com/your-org/your-repo.git
# %cd your-repo

# Option 2: if you uploaded the project zip manually, set the working directory here
import os
from pathlib import Path

PROJECT_DIR = Path('/content/Fine-Tuning')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
%cd /content
print('Set PROJECT_DIR to', PROJECT_DIR)


In [ ]:
%cd /content/Fine-Tuning
!python -m pip install --upgrade pip
!pip install -r requirements.txt


In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 1. Generate Dataset

Default size is 4,000 examples with 12% refusals and an 80/10/10 split.


In [ ]:
%cd /content/Fine-Tuning
!python prepare_dataset.py --num-examples 4000 --output-dir data/processed


In [ ]:
import json
from pathlib import Path

sample_path = Path('data/processed/train.jsonl')
with sample_path.open('r', encoding='utf-8') as handle:
    first = json.loads(handle.readline())
first


## 2. Train SFT QLoRA Adapter

This uses the project config:
- Qwen2.5-3B-Instruct
- 4-bit NF4 QLoRA
- LoRA rank 16
- batch size 4
- gradient accumulation 4
- 3 epochs
- learning rate 2e-4


In [ ]:
%cd /content/Fine-Tuning
!python train_sft.py \
  --train-file data/processed/train.jsonl \
  --val-file data/processed/val.jsonl \
  --output-dir outputs/sft-qwen2.5-3b-json


## 3. Evaluate Baseline and SFT

Run on the held-out test split and compare metrics.


In [ ]:
%cd /content/Fine-Tuning
!python evaluate.py \
  --dataset-file data/processed/test.jsonl \
  --output-file results/baseline_eval.json


In [ ]:
%cd /content/Fine-Tuning
!python evaluate.py \
  --dataset-file data/processed/test.jsonl \
  --adapter-path outputs/sft-qwen2.5-3b-json \
  --output-file results/sft_eval.json


In [ ]:
import json
from pathlib import Path

for name in ['baseline_eval.json', 'sft_eval.json']:
    path = Path('results') / name
    print('\n' + name)
    data = json.loads(path.read_text(encoding='utf-8'))
    print({k: v for k, v in data.items() if k != 'predictions'})


## 4. Generate DPO Preference Pairs

This cell uses the SFT adapter to sample four candidates per prompt and asks GPT-4o-mini to choose the best and worst candidate.


In [ ]:
import os

# Set this manually if needed.
# os.environ['OPENAI_API_KEY'] = 'sk-...'

print('OPENAI_API_KEY present:', bool(os.environ.get('OPENAI_API_KEY')))


In [ ]:
%cd /content/Fine-Tuning
!python generate_dpo_pairs.py \
  --dataset-file data/processed/test.jsonl \
  --adapter-path outputs/sft-qwen2.5-3b-json \
  --output-dir data/dpo_pairs \
  --num-candidates 4


## 5. Train DPO Adapter

This runs DPO on top of the SFT checkpoint with `beta=0.1`.


In [ ]:
%cd /content/Fine-Tuning
!python train_dpo.py \
  --sft-checkpoint outputs/sft-qwen2.5-3b-json \
  --preference-file data/dpo_pairs/train.jsonl \
  --val-file data/dpo_pairs/val.jsonl \
  --output-dir outputs/dpo-qwen2.5-3b-json \
  --beta 0.1


## 6. Evaluate DPO


In [ ]:
%cd /content/Fine-Tuning
!python evaluate.py \
  --dataset-file data/processed/test.jsonl \
  --adapter-path outputs/dpo-qwen2.5-3b-json \
  --output-file results/dpo_eval.json


In [ ]:
import json
from pathlib import Path

def load_metrics(name):
    data = json.loads((Path('results') / name).read_text(encoding='utf-8'))
    return {
        'json_validity_rate': data['json_validity_rate'],
        'exact_match_accuracy': data['exact_match_accuracy'],
        'field_level_f1': data['field_level_f1'],
        'refusal_correctness': data['refusal_correctness'],
    }

baseline = load_metrics('baseline_eval.json')
sft = load_metrics('sft_eval.json')
dpo = load_metrics('dpo_eval.json')

print('Baseline:', baseline)
print('SFT:', sft)
print('DPO:', dpo)
print('\nSFT delta over baseline:')
for k in baseline:
    print(k, round(sft[k] - baseline[k], 2))
print('\nDPO delta over SFT:')
for k in sft:
    print(k, round(dpo[k] - sft[k], 2))


## 7. Optional: Launch FastAPI in Colab

This is optional and mainly useful for a quick smoke test.


In [ ]:
# %env BASE_MODEL=Qwen/Qwen2.5-3B-Instruct
# %env ADAPTER_PATH=outputs/dpo-qwen2.5-3b-json
# !uvicorn serve:app --host 0.0.0.0 --port 8000


## 8. Save Artifacts

Zip results and adapters so you can download them from Colab.


In [ ]:
%cd /content/Fine-Tuning
!zip -r structured-json-ft-artifacts.zip results outputs data/dpo_pairs
